# Пример 5. Граф исполнения на LangChain / LangGraph

**Модуль I · Тема 3** — обзор программных каркасов: вершины, рёбра, состояние, граф исполнения

В примере 4 цикл агента был написан вручную. Здесь собирается **тот же самый агент**, но средствами каркаса. Цель — увидеть, что каркас не приносит новой магии: он оформляет уже знакомый цикл как **граф исполнения**.

Сравнение будет предметным: одинаковый домен, одинаковые инструменты, одинаковая роль.

### Две библиотеки — общая картина (если видите их впервые)

- **LangChain** — строительные блоки: модель (`ChatOpenAI`), инструменты (декоратор `@tool`), сообщения (`SystemMessage` и др.).
- **LangGraph** — оркестратор: описывает работу агента как **граф** из трёх сущностей:
  - **вершины (nodes)** — шаги (вызвать модель, исполнить инструмент);
  - **рёбра (edges)** — переходы, в том числе **условные** (ветвление);
  - **состояние (state)** — данные, которые вершины читают и дополняют.

### Что показано
1. Инструменты через `@tool`.
2. Сборка графа вручную: вершины, рёбра, условное ребро.
3. Пошаговое наблюдение цикла (`stream`).
4. Сопоставление графа с ручным циклом из примера 4.

### Доступ к модели
Параметры — из переменных окружения `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`.

## 1. Модель: класс `ChatOpenAI`

`ChatOpenAI` из пакета `langchain_openai` — обёртка над chat-эндпоинтом любой OpenAI-совместимой модели.

| Аргумент | Что задаёт |
|----------|-----------|
| `model` | идентификатор модели |
| `base_url` | адрес эндпоинта |
| `api_key` | ключ (из окружения, в код не вставляется) |
| `temperature` | `0` — максимально детерминированно |

Методы, которые понадобятся: `llm.invoke(messages)` — один вызов; `llm.bind_tools(tools)` — вернуть модель, «знающую» о наборе инструментов.

In [1]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=os.environ.get("LLM_MODEL", "openai/gpt-4.1-mini"),
                 base_url=os.environ.get("LLM_BASE_URL"),
                 api_key=os.environ.get("LLM_API_KEY"),
                 temperature=0)
print("Модель:", llm.model_name)

Модель: openai/gpt-4.1-mini


## 2. Инструменты через декоратор `@tool`

В примере 4 схему инструмента (JSON Schema) мы писали руками. Декоратор `@tool` собирает её **автоматически из самой функции**:

| Что нужно модели | Откуда берётся |
|------------------|----------------|
| имя | имя функции |
| описание (по нему модель решает, когда звать) | докстринг |
| схема аргументов | аннотации типов |

Сравните объём кода с ячейкой `TOOLS_SPEC` из примера 4 — это основная экономия, которую даёт каркас.

In [2]:
from langchain_core.tools import tool

ORDERS = {
    "ORD-1002": {"status": "отправлен", "items": "USB-C хаб ×2", "total": 4980},
    "ORD-1003": {"status": "в сборке",  "items": "наушники ×1",  "total": 4990},
}
POLICIES = {
    "возврат":  "Возврат в течение 14 дней с момента получения при сохранении упаковки.",
    "доставка": "Стандартная доставка 3–5 рабочих дней.",
}

@tool
def get_order(order_id: str) -> dict:
    """Получить информацию о заказе по его идентификатору (например ORD-1002)."""
    if order_id not in ORDERS:
        return {"error": "not_found", "order_id": order_id}
    return {"order_id": order_id, **ORDERS[order_id]}

@tool
def get_policy(topic: str) -> dict:
    """Получить правила магазина по теме: возврат или доставка."""
    if topic not in POLICIES:
        return {"error": "unknown_topic", "available": list(POLICIES)}
    return {"topic": topic, "text": POLICIES[topic]}

TOOLS = [get_order, get_policy]
print("Инструмент:", get_order.name)
print("Описание  :", get_order.description)
print("Схема арг.:", get_order.args)

Инструмент: get_order
Описание  : Получить информацию о заказе по его идентификатору (например ORD-1002).
Схема арг.: {'order_id': {'title': 'Order Id', 'type': 'string'}}


## 3. Сборка графа исполнения

Разберём каждое имя, которое встретится в коде ниже.

**Состояние.**
- `MessagesState` — готовая схема состояния: словарь с полем `messages`. У поля встроенный **reducer**: когда вершина возвращает `{"messages": [...]}`, LangGraph **добавляет** сообщения к списку, а не перезаписывает его. Благодаря этому история диалога накапливается сама (в примере 4 мы делали это вручную через `messages.append`).

**Вершины.** Вершина — обычная функция `состояние → частичное обновление состояния`.
- `agent_node` — наша вершина: вызывает модель с привязанными инструментами.
- `ToolNode(TOOLS)` — **готовая** вершина: читает запросы на вызовы из последнего сообщения, исполняет нужные функции и возвращает результаты. Это замена всего блока `for tc in msg.tool_calls` из примера 4.

**Сборка.**
- `StateGraph(MessagesState)` — строитель графа поверх схемы состояния;
- `.add_node("имя", функция)` — зарегистрировать вершину;
- `START`, `END` — служебные метки входа и выхода;
- `.add_edge(A, B)` — безусловный переход;
- `.add_conditional_edges("agent", tools_condition)` — **условное** ребро: готовый маршрутизатор `tools_condition` смотрит на ответ модели и направляет в `"tools"`, если запрошен инструмент, иначе в `END`. Это ровно тот `if not msg.tool_calls`, что был в примере 4;
- `.compile()` — превратить описание в исполняемый граф.

In [3]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import SystemMessage

SYSTEM_PROMPT = """Ты — агент поддержки интернет-магазина «ШопБот».

ЦЕЛЬ. Отвечать на вопросы клиентов о статусе заказов и правилах магазина.

ОГРАНИЧЕНИЯ И ЗАПРЕТЫ.
- Не сообщай данных, которых нет в результатах инструментов.
- Не обещай сроки, скидки и компенсации.
- Не отвечай на вопросы вне тематики магазина.

СТИЛИСТИКА. Русский язык, вежливо, не более 3 предложений; указывай идентификатор заказа.

ПОЛИТИКА ПРИНЯТИЯ РЕШЕНИЙ.
- Нужны данные — вызови инструмент, не отвечай по памяти.
- Ошибка not_found — сообщи, что заказ не найден, попроси проверить номер.
- Вопрос вне тематики — вежливо откажи."""

llm_with_tools = llm.bind_tools(TOOLS)
SYS = SystemMessage(SYSTEM_PROMPT)

def agent_node(state: MessagesState):
    """Вершина «модель»: получает историю, возвращает ответ модели."""
    return {"messages": [llm_with_tools.invoke([SYS] + state["messages"])]}

g = StateGraph(MessagesState)
g.add_node("agent", agent_node)
g.add_node("tools", ToolNode(TOOLS))
g.add_edge(START, "agent")                        # вход -> модель
g.add_conditional_edges("agent", tools_condition) # модель -> tools или END
g.add_edge("tools", "agent")                      # инструменты -> снова модель (замыкаем цикл)
graph = g.compile()

print("Граф собран. Вершины:", list(graph.get_graph().nodes))

Граф собран. Вершины: ['__start__', 'agent', 'tools', '__end__']


### Схема графа

```
        START
          │
          ▼
      ┌───────┐   есть запрос инструмента
      │ agent │ ──────────────────────────┐
      └───────┘                           ▼
          ▲                           ┌───────┐
          │      результаты           │ tools │
          └───────────────────────────└───────┘
          │
          │ ответ готов
          ▼
         END
```

**Сопоставление с примером 4:**

| Граф (пример 5) | Ручной цикл (пример 4) |
|-----------------|------------------------|
| вершина `agent` | вызов `client.chat.completions.create(...)` |
| вершина `tools` (`ToolNode`) | блок `for tc in msg.tool_calls: ...` |
| условное ребро `tools_condition` | `if not msg.tool_calls: return ...` |
| ребро `tools → agent` | продолжение цикла `for step in range(...)` |
| reducer у `MessagesState` | `messages.append(...)` |
| `.compile()` + `.invoke()` | сам цикл `for` |

## 4. Запуск с наблюдением шагов

Метод `.stream(вход)` выполняет граф, отдавая обновления **по мере прохождения вершин**. Это аналог печати трассы из примера 4 — видно тот же цикл «модель → инструменты → модель».

In [4]:
query = "Что с заказом ORD-1002 и как оформить возврат?"

for chunk in graph.stream({"messages": [("user", query)]}):
    for node, update in chunk.items():
        last = update["messages"][-1]
        if getattr(last, "tool_calls", None):
            names = ", ".join(tc["name"] for tc in last.tool_calls)
            print(f"[{node}] модель запросила инструменты: {names}")
        elif last.type == "tool":
            print(f"[{node}] результат {last.name}: {str(last.content)[:80]}")
        else:
            print(f"[{node}] финальный ответ: {last.content[:120]}")

[agent] модель запросила инструменты: get_order, get_policy
[tools] результат get_policy: {"topic": "возврат", "text": "Возврат в течение 14 дней с момента получения при 


[agent] финальный ответ: Заказ ORD-1002 со статусом "отправлен", в нем USB-C хаб ×2 на сумму 4980 рублей. Возврат возможен в течение 14 дней с мо


## 5. Проверка на тех же трёх сценариях

Те же сценарии, что и в примере 4: успех, отказ, деградация. Поведение должно совпадать — роль и инструменты те же.

In [5]:
def run(query: str):
    out = graph.invoke({"messages": [("user", query)]})
    msgs = out["messages"]
    tool_calls = sum(1 for m in msgs if m.type == "tool")
    return msgs[-1].content, tool_calls, len(msgs)

for name, q in [("успех",      "Что с заказом ORD-1002 и как оформить возврат?"),
                ("отказ",      "Напиши функцию быстрой сортировки на Python."),
                ("деградация", "Какой статус у заказа ORD-99999?")]:
    answer, tools_used, n = run(q)
    print(f"--- {name} (инструментов: {tools_used}, сообщений в трассе: {n}) ---")
    print(answer[:220], "\n")

--- успех (инструментов: 2, сообщений в трассе: 5) ---
Заказ ORD-1002 со статусом "отправлен", в нем USB-C хаб ×2 на сумму 4980 рублей. Возврат возможен в течение 14 дней с момента получения при сохранении упаковки. Если нужна помощь с возвратом, обращайтесь. 



--- отказ (инструментов: 0, сообщений в трассе: 2) ---
Извините, но я могу помочь только с вопросами, связанными с заказами и правилами магазина. Если у вас есть вопросы по этим темам, пожалуйста, задайте. 



--- деградация (инструментов: 1, сообщений в трассе: 4) ---
Заказ с идентификатором ORD-99999 не найден. Пожалуйста, проверьте правильность номера заказа. 



## 6. Полная трасса из состояния графа

Каркас накапливает всю историю в состоянии, поэтому трасса извлекается из результата — отдельно её собирать не нужно.

In [6]:
import pandas as pd

out = graph.invoke({"messages": [("user", "Что с заказом ORD-1002 и как оформить возврат?")]})

rows = []
for i, m in enumerate(out["messages"], 1):
    if m.type == "human":
        kind, detail = "запрос пользователя", m.content
    elif m.type == "tool":
        kind, detail = f"результат {m.name}", str(m.content)
    elif getattr(m, "tool_calls", None):
        kind = "модель: запрос инструментов"
        detail = ", ".join(f"{tc['name']}({tc['args']})" for tc in m.tool_calls)
    else:
        kind, detail = "модель: финальный ответ", m.content
    rows.append({"№": i, "сообщение": kind, "содержание": detail[:70]})

print(pd.DataFrame(rows).to_string(index=False))

 №                   сообщение                                                             содержание
 1         запрос пользователя                         Что с заказом ORD-1002 и как оформить возврат?
 2 модель: запрос инструментов  get_order({'order_id': 'ORD-1002'}), get_policy({'topic': 'возврат'})
 3         результат get_order {"order_id": "ORD-1002", "status": "отправлен", "items": "USB-C хаб ×2
 4        результат get_policy {"topic": "возврат", "text": "Возврат в течение 14 дней с момента полу
 5     модель: финальный ответ Заказ ORD-1002 со статусом "отправлен", в нем 2 USB-C хаба на сумму 49


## Итоги

- **Каркас не меняет сути.** Граф исполнения — то же самое, что цикл из примера 4: вершина `agent` — вызов модели, `tools` — исполнение инструментов, условное ребро — проверка «нужен ли инструмент».
- **Что каркас берёт на себя:** схемы инструментов (`@tool`), разбор и исполнение вызовов (`ToolNode`), накопление истории (reducer состояния), маршрутизацию (`tools_condition`).
- **Чем платим:** цикл перестаёт быть виден в коде. Поэтому в курсе сначала пишется руками (пример 4), и только потом берётся каркас.
- **Что не изменилось:** роль по-прежнему задаётся системной инструкцией, а корректный отказ остаётся правильным результатом.

### Справочник использованных имён

| Имя | Библиотека | Что делает |
|-----|-----------|-----------|
| `ChatOpenAI(...)` | LangChain | объект модели |
| `llm.bind_tools(tools)` | LangChain | модель, умеющая запрашивать инструменты |
| `@tool` | LangChain | функция → инструмент (имя/описание/схема автоматически) |
| `SystemMessage` | LangChain | сообщение уровня `system` |
| `StateGraph(MessagesState)` | LangGraph | строитель графа |
| `MessagesState` | LangGraph | схема состояния с накоплением сообщений |
| `.add_node / .add_edge` | LangGraph | вершины и безусловные переходы |
| `.add_conditional_edges` | LangGraph | ветвление по функции-маршрутизатору |
| `START`, `END` | LangGraph | вход и выход графа |
| `ToolNode(tools)` | LangGraph | готовая вершина исполнения инструментов |
| `tools_condition` | LangGraph | маршрутизатор «нужен инструмент? → tools, иначе END» |
| `.compile()` / `.invoke()` / `.stream()` | LangGraph | сборка и запуск (целиком / пошагово) |

**В лабораторной работе ([КИМ-1.3](../../kim-lab-03.md), часть 3)** этот граф строится для вашей роли, а часть 2 требует сначала реализовать цикл вручную.